# Aula 19 · Hands-on: monitoramento e reparo de tarefa

## O que é
O **Repair run** reexecuta **só a tarefa que falhou e as que dependem dela**, aproveitando tudo o que já deu certo.

## Por que importa
Reexecutar um job inteiro por causa de uma tarefa **custa tempo e dinheiro** e, se o pipeline não for idempotente, pode **duplicar dados**.

## Como aplicar
### 1. Provocar uma falha controlada
O notebook `Projeto/03_gold` tem o parâmetro `simular_falha`. Com valor `sim`, ele falha de propósito.

1. No job, **Job parameters** → adicione `simular_falha` = `nao` (valor padrão).
2. **Run now with different parameters** → `simular_falha` = `sim`.
3. Espere a execução terminar. A `gold` fica **vermelha** depois das 2 retentativas, e `risco_alto` e `alerta_risco` não rodam. `registrar_execucao` roda mesmo assim (Run if: All done), por isso a execução termina como **Succeeded with failures** (aula 18).

### 2. Investigar a causa
1. Abra a execução → clique na tarefa `gold`.
2. Leia o **erro** no topo da saída: `Falha simulada: a tabela de referência de clientes não está disponível...`
3. Role até a célula que falhou: o notebook mostra o *stack trace*.
4. Confira o e-mail de falha da tarefa `gold` (configurado na aula 18).

> Roteiro de investigação para a vida real: **1)** qual tarefa falhou? **2)** qual a mensagem de erro? **3)** em qual célula? **4)** o que mudou desde a última execução com sucesso (código, dado, parâmetro)?

### 3. Corrigir e reparar
1. Na execução que falhou → **Repair run** (canto superior direito).
2. Repare que **só** `gold` e as tarefas depois dela estão selecionadas. `preparar_ambiente`, `bronze`, `silver` e `validar_qualidade` **não** vão rodar de novo.
3. Em **Job parameters**, altere `simular_falha` para `nao`.
4. **Repair run**.

### 4. Confirmar o resultado
- A mesma execução agora mostra o histórico: **Run #1 (falhou) → Repair #1 (sucesso)**.
- O DAG fica todo verde.

In [ ]:
%sql
DESCRIBE HISTORY growdev.gold.resumo_clientes LIMIT 3

## Checklist da aula
- [ ] Provoquei uma falha controlada
- [ ] Encontrei a causa pela saída e pelo erro da tarefa
- [ ] Usei o Repair run para reexecutar só o que faltava
- [ ] Confirmei o sucesso no histórico da execução